# 端到端 capstone：从 canonical 问题到可追溯回答

本 Notebook 把[端到端验收协议](端到端验收协议.md)落成一个小规模、可重复执行的闭环：canonical regression 问题 → BM25 检索 → 上下文预算 → 真实 `glm-4-flash` 回答 → `evidence_id` 引用与逐字 quote hydrate → 事后 qrels 检查或拒答检查。

样本固定为 3 道 answerable 题（其中 2 道是多证据题）和 1 道 canonical unanswerable 题。证据池只保留 canonical `evidence_type=quote` 原文，不读 C2 训练候选；本页不是 33 种方法的统一 benchmark。缺少密钥、依赖、证据或模型返回不符合契约时直接报错。

## 运行边界与验收协议

- 检索前只把 `query_id` 和 `text` 投影给检索器；`qrels`、`reference_answer`、`expected_pages` 不参与排序、截断或提示词。
- 检索器是教程核心 benchmark 使用的 BM25；`top_k=8`，上下文预算为 1,800 个字符，并且只按完整 evidence quote 逐条纳入。
- answerable 题必须由 `glm-4-flash` 只根据上下文作答，并返回实际上下文中的 `evidence_id`；Notebook 再从 canonical evidence 逐字 hydrate quote。
- 四道题都在同一检索和预算之后调用真实 `glm-4-flash`；模型只依据 QUESTION+CONTEXT 返回 `answerable` 或 `insufficient`。模型输出之后才加载 query metadata/qrels，检查 essential evidence 覆盖、引用身份和拒答契约。

In [1]:
from pathlib import Path
import json
import sys

COURSE_CANDIDATES = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
COURSE_ROOT = next((path for path in COURSE_CANDIDATES if path.name == "C7 高级 RAG 技巧" and (path / "common").is_dir()), None)
if COURSE_ROOT is None:
    raise RuntimeError(f"当前路径不在 C7 教程目录：{Path.cwd()}")
sys.path.insert(0, str(COURSE_ROOT))

from common.dataset import (
    DATASET_ROOT,
    read_jsonl,
)
from common.eval_utils import build_bm25_chunk_search, emit_tutorial_audit
from common.nontraining_utils import RAG_LLM_MODEL, llm_call
INSUFFICIENT_ANSWER = "资料不足，无法根据提供的上下文回答。"

if RAG_LLM_MODEL != "glm-4-flash":
    raise RuntimeError("capstone 固定要求真实 glm-4-flash；请不要用环境变量切换模型")

CASE_IDS = (
    "model_selection_with_intro_scope",
    "cross_validation_reliability",
    "model_evaluation_and_macro_micro",
    "book_evidence_boundary",
)
TOP_K = 8
CONTEXT_BUDGET = 1800
print("model =", RAG_LLM_MODEL, "; cases =", len(CASE_IDS), "; context_budget =", CONTEXT_BUDGET)

model = glm-4-flash ; cases = 4 ; context_budget = 1800


## 1. 检索：只接收问题文字和 canonical evidence

这里直接从 canonical `queries.jsonl` 和 `evidence.jsonl` 做字段投影，只保留问题身份、问题文字以及 `evidence_id/page/quote`；不会加载 C2 的 `qa_candidates` 或 `finetune_pairs`。排序器实际看到的字段不含任何 qrels 或参考答案。

In [2]:
# 检索前的输入边界：明确只构造 query_id/text 投影；不调用会加载
# C2 候选的完整数据集读取器。
query_rows = read_jsonl(DATASET_ROOT / "queries.jsonl")
query_only = {
    str(row["query_id"]): str(row["text"])
    for row in query_rows
    if row.get("query_id") in CASE_IDS
}
if set(query_only) != set(CASE_IDS):
    raise AssertionError("固定 capstone 问题缺失")

all_evidence = read_jsonl(DATASET_ROOT / "evidence.jsonl")
canonical_evidence = [
    {
        "evidence_id": str(row["evidence_id"]),
        "page": int(row["page"]),
        "quote": str(row["quote"]),
    }
    for row in all_evidence
    if row.get("evidence_type") == "quote"
]
if not canonical_evidence:
    raise ValueError("canonical quote evidence 为空")
if any(row["evidence_id"].startswith("evi_ft_") for row in canonical_evidence):
    raise AssertionError("C2 训练候选混入 capstone evidence 池")

search = build_bm25_chunk_search(
    [
        {"chunk_id": row["evidence_id"], "pages": [row["page"]], "text": row["quote"]}
        for row in canonical_evidence
    ]
)
retrieved = {case_id: search(query, top_k=TOP_K) for case_id, query in query_only.items()}
if any(not rows for rows in retrieved.values()):
    raise AssertionError("至少有一道题没有检索结果")

retrieved_ids = {
    case_id: [item.chunk_id for item in rows]
    for case_id, rows in retrieved.items()
}
for case_id in CASE_IDS:
    print(f"\n[{case_id}] {query_only[case_id]}")
    for rank, item in enumerate(retrieved[case_id], start=1):
        print(f"  {rank:>2}. {item.chunk_id} | p.{item.pages[0]} | score={item.score:.3f}")

print("\n检索阶段完成：排序输入只有 query text 与 canonical quote；qrels 尚未读取。")


[model_selection_with_intro_scope] 《南瓜书》绪论里说，机器学习算法之间有没有绝对更好的一个？
   1. evi_065c349a8cbf | p.17 | score=38.044
   2. evi_4534a9518d9a | p.15 | score=24.274
   3. evi_718a2d52ed5c | p.18 | score=23.856
   4. evi_451af37f42ba | p.118 | score=22.086
   5. evi_32c0b230bb7c | p.17 | score=20.795
   6. evi_9b2016615541 | p.88 | score=20.298
   7. evi_8845538704bf | p.16 | score=19.029
   8. evi_3bc6b53ddcaf | p.100 | score=16.856

[cross_validation_reliability] 交叉验证法为什么比单次留出法更可靠？
   1. evi_718a2d52ed5c | p.18 | score=27.792
   2. evi_df934cb5bae0 | p.18 | score=27.755
   3. evi_9a292ab3d933 | p.19 | score=26.416
   4. evi_5f5d3dc01c2d | p.19 | score=26.135
   5. evi_833c8fa10a30 | p.19 | score=26.049
   6. evi_4c24466586af | p.18 | score=22.653
   7. evi_40e63a429c30 | p.19 | score=16.742
   8. evi_24f3ee133e62 | p.39 | score=10.452

[model_evaluation_and_macro_micro] “模型评估与选择”是什么？为什么类别不平衡时要区分宏平均和微平均？
   1. evi_91d0b1e7d0ff | p.18 | score=33.879
   2. evi_ffd64ad64243 | p.21 | score=32.517
 

## 2. 上下文预算：按完整证据逐条截断

上下文预算不是把 quote 从中间切断，而是按 BM25 顺序加入完整 evidence。这样后续引用可以用 `evidence_id` 找回 canonical 原文；若第一条证据本身超过预算，直接失败而不静默截断。

In [3]:
evidence_by_id = {row["evidence_id"]: row for row in canonical_evidence}

def context_block(row):
    return f"[evidence_id={row['evidence_id']} page={row['page']}]\n{row['quote']}"

def apply_context_budget(items, budget=CONTEXT_BUDGET):
    separator = "\n\n"
    selected = []
    used = 0
    for item in items:
        row = evidence_by_id[item.chunk_id]
        block = context_block(row)
        if len(block) > budget and not selected:
            raise ValueError(f"首条 evidence 超过上下文预算：{item.chunk_id}")
        added = len(block) + (len(separator) if selected else 0)
        if used + added > budget:
            break
        selected.append(row)
        used += added
    if not selected:
        raise ValueError("上下文预算后没有可用 evidence")
    return selected, used

context_rows = {}
context_text = {}
for case_id, items in retrieved.items():
    rows, used = apply_context_budget(items)
    context_rows[case_id] = rows
    context_text[case_id] = "\n\n".join(context_block(row) for row in rows)
    print(f"{case_id}: {len(rows)} 条 evidence，{used}/{CONTEXT_BUDGET} chars")
    if len(context_text[case_id]) != used:
        raise AssertionError("上下文预算统计与实际 join 长度不一致")
    if len(context_text[case_id]) > CONTEXT_BUDGET:
        raise AssertionError("实际拼接后的上下文超过字符预算")
    if not set(retrieved_ids[case_id]) >= {row["evidence_id"] for row in rows}:
        raise AssertionError("上下文 evidence 不在检索结果中")

cross_validation_reliability: 8 条 evidence，1251/1800 chars
book_evidence_boundary: 8 条 evidence，1637/1800 chars
model_evaluation_and_macro_micro: 8 条 evidence，1130/1800 chars
model_selection_with_intro_scope: 7 条 evidence，1554/1800 chars


## 3. 真实回答：模型只返回身份引用，quote 由 Notebook hydrate

本单元不读取 query metadata、qrels 或参考答案；四道题统一调用真实 `glm-4-flash`。模型只能根据 QUESTION+CONTEXT 返回 `answerable` 或 `insufficient`，并列出上下文中的 `evidence_id`；Notebook 不让模型自行改写 quote。

In [4]:
def generation_prompt(question, context):
    return (
        "你是一个严格的资料问答器。只能依据 CONTEXT 回答 QUESTION，不得使用外部知识。status 只能是 answerable 或 insufficient。\n"
        "请只输出一个 JSON 对象，不要 Markdown 代码围栏，格式必须是："
        '{"status":"answerable","answer":"...","citations":[{"evidence_id":"..."}]}\n'
        "如果 CONTEXT 不足以回答，必须返回 status=insufficient、answer=“资料不足，无法根据提供的上下文回答。”、citations=[]，不得添加其他内容。"
        "answerable 时 citations 必须列出支持答案的全部 evidence_id；只能从 CONTEXT 复制身份，"
        "不要输出 quote 字段，Notebook 会按 evidence_id 从 canonical evidence 逐字 hydrate。\n\n"
        f"QUESTION:\n{question}\n\nCONTEXT:\n{context}"
    )

def validate_model_json(raw, allowed_ids):
    # 允许模型 SDK 常见的完整 ```json ... ``` 传输外壳；外壳内仍必须是
    # 严格 JSON，任何其他前后缀、修复或默认答案都直接失败。
    payload = str(raw).strip()
    if payload.startswith("```json") and payload.endswith("```"):
        payload = payload[len("```json"):-3].strip()
    try:
        parsed = json.loads(payload)
    except json.JSONDecodeError as exc:
        raise ValueError(f"模型没有返回严格 JSON：{raw!r}") from exc
    if not isinstance(parsed, dict):
        raise ValueError(f"模型返回的不是 JSON 对象：{raw!r}")
    if set(parsed) != {"status", "answer", "citations"}:
        raise ValueError(f"模型 JSON 字段不严格：{parsed!r}")
    if parsed.get("status") not in {"answerable", "insufficient"}:
        raise ValueError(f"模型 status 必须为 answerable 或 insufficient：{parsed!r}")
    if not isinstance(parsed.get("answer"), str) or not parsed["answer"].strip():
        raise ValueError(f"模型 answer 为空：{parsed!r}")
    citations = parsed.get("citations")
    if not isinstance(citations, list):
        raise ValueError(f"模型 citations 不是列表：{parsed!r}")
    if parsed["status"] == "insufficient" and parsed["answer"] != INSUFFICIENT_ANSWER:
        raise ValueError(f"insufficient answer 必须是简洁资料不足声明：{parsed!r}")
    if parsed["status"] == "insufficient" and citations:
        raise ValueError(f"insufficient 必须严格返回空 citations：{parsed!r}")
    if parsed["status"] == "answerable" and not citations:
        raise ValueError(f"answerable 必须返回 citations：{parsed!r}")
    ids = []
    for citation in citations:
        if not isinstance(citation, dict) or set(citation) != {"evidence_id"}:
            raise ValueError(f"citation 必须只有 evidence_id：{parsed!r}")
        evidence_id = citation["evidence_id"]
        if evidence_id not in allowed_ids:
            raise ValueError(f"模型引用了不在上下文中的 evidence：{evidence_id}")
        if evidence_id in ids:
            raise ValueError(f"模型重复引用 evidence：{evidence_id}")
        ids.append(evidence_id)
    parsed["citation_ids"] = ids
    return parsed

raw_model = {}
responses = {}
for case_id in CASE_IDS:
    raw = llm_call(
        generation_prompt(query_only[case_id], context_text[case_id]),
        max_tokens=700,
    )
    raw_model[case_id] = raw
    parsed = validate_model_json(raw, {row["evidence_id"] for row in context_rows[case_id]})
    hydrated = [
        {
            "evidence_id": evidence_id,
            "page": evidence_by_id[evidence_id]["page"],
            "quote": evidence_by_id[evidence_id]["quote"],
        }
        for evidence_id in parsed["citation_ids"]
    ]
    responses[case_id] = {
        "status": parsed["status"],
        "answer": parsed["answer"],
        "citation_ids": parsed["citation_ids"],
        "hydrated_citations": hydrated,
        "model_called": True,
    }
    print(f"{case_id}: glm-4-flash 返回 {parsed['status']}，{len(hydrated)} 条引用")

model_selection_with_intro_scope: glm-4-flash 返回 answerable，2 条引用


cross_validation_reliability: glm-4-flash 返回 answerable，7 条引用


model_evaluation_and_macro_micro: glm-4-flash 返回 answerable，2 条引用


book_evidence_boundary: glm-4-flash 返回 insufficient，0 条引用


## 4. 独立语义检查：逐条判断结论是否被 quote 蕴含

只检查引用 ID 还不够：恶意回答可以保留合法 ID，却把结论改成“CUDA99”或“交叉验证无价值”。这里再用一次独立的真实 `glm-4-flash` 评审模型；它只看到问题、回答状态、回答文字和已经 hydrate 的 quote，逐条返回 `supported/contradicted/not_found` 及实际 evidence_id。只要存在非 `supported` 的主要结论，capstone 就失败。该评审仍是模型辅助检查，不是人工事实证明，只有 4 道固定题，不能外推到一般任务。

In [5]:
SEMANTIC_JUDGE_PROMPT = (
    "你是独立的回答语义审计器，不是回答者。只能依据 QUESTION、ANSWER 和 CITED_EVIDENCE 判断，不能使用外部知识。\n"
    "把 ANSWER 拆成不超过 5 条主要可验证结论。每条严格返回 claim、relation、evidence_ids；"
    "relation 只能是 supported（对应 quote 直接蕴含结论）、contradicted（对应 quote 明确反驳）或 not_found（引用 quote 既未蕴含也未反驳）。\n"
    "supported/contradicted 必须列出实际支持或反驳该结论的 evidence_id，not_found 的 evidence_ids 必须为空。"
    "只有所有主要结论都 supported 且确实回应 QUESTION 时 verdict 才能是 pass；否则必须是 fail。\n"
    "若 STATUS=insufficient，只有 ANSWER 恰好是资料不足声明时才允许 claims=[]、verdict=pass；不能夹带任何答案或推测。\n"
    "只输出严格 JSON：{{\"claims\":[{{\"claim\":\"...\",\"relation\":\"supported|contradicted|not_found\",\"evidence_ids\":[\"...\"]}}],\"verdict\":\"pass|fail\",\"reason\":\"...\"}}。\n\n"
    "QUESTION: {question}\nSTATUS: {status}\nANSWER: {answer}\nCITED_EVIDENCE:\n{evidence}"
)

def parse_strict_json(raw):
    payload = str(raw).strip()
    if payload.startswith("```json") and payload.endswith("```"):
        payload = payload[len("```json"):-3].strip()
    try:
        value = json.loads(payload)
    except json.JSONDecodeError as exc:
        raise ValueError(f"语义评审没有返回严格 JSON：{raw!r}") from exc
    if not isinstance(value, dict):
        raise ValueError(f"语义评审 JSON 不是对象：{raw!r}")
    return value

def validate_semantic_judge(raw, allowed_ids, generation_status):
    value = parse_strict_json(raw)
    if set(value) != {"claims", "verdict", "reason"}:
        raise ValueError(f"语义评审 JSON 字段不严格：{value!r}")
    if value["verdict"] not in {"pass", "fail"} or not isinstance(value["reason"], str) or not value["reason"].strip():
        raise ValueError(f"语义评审 verdict/reason 无效：{value!r}")
    claims = value["claims"]
    if not isinstance(claims, list):
        raise ValueError(f"语义评审 claims 不是列表：{value!r}")
    if generation_status == "insufficient":
        if claims or value["verdict"] != "pass":
            raise ValueError(f"insufficient 语义检查必须 claims=[] 且 pass：{value!r}")
        return value
    if generation_status != "answerable" or not claims:
        raise ValueError(f"answerable 语义检查必须有 claims：{value!r}")
    for item in claims:
        if not isinstance(item, dict) or set(item) != {"claim", "relation", "evidence_ids"}:
            raise ValueError(f"语义评审 claim 字段不严格：{item!r}")
        if not isinstance(item["claim"], str) or not item["claim"].strip():
            raise ValueError(f"语义评审 claim 为空：{item!r}")
        if item["relation"] not in {"supported", "contradicted", "not_found"}:
            raise ValueError(f"语义评审 relation 非法：{item!r}")
        evidence_ids = item["evidence_ids"]
        if not isinstance(evidence_ids, list) or len(evidence_ids) != len(set(evidence_ids)):
            raise ValueError(f"语义评审 evidence_ids 必须是无重复列表：{item!r}")
        if not set(evidence_ids) <= set(allowed_ids):
            raise ValueError(f"语义评审引用了未 hydrate 的 evidence：{item!r}")
        if item["relation"] == "not_found" and evidence_ids:
            raise ValueError(f"not_found 的 evidence_ids 必须为空：{item!r}")
        if item["relation"] in {"supported", "contradicted"} and not evidence_ids:
            raise ValueError(f"{item['relation']} 必须列出 evidence_ids：{item!r}")
    return value

semantic_results = {}
for case_id in CASE_IDS:
    response = responses[case_id]
    cited_evidence = "\n\n".join(
        context_block(row) for row in response.get("hydrated_citations", [])
    ) or "(none)"
    raw = llm_call(
        SEMANTIC_JUDGE_PROMPT.format(
            question=query_only[case_id],
            status=response["status"],
            answer=response["answer"],
            evidence=cited_evidence,
        ),
        max_tokens=900,
    )
    parsed = validate_semantic_judge(
        raw, set(response.get("citation_ids", [])), response["status"]
    )
    semantic_results[case_id] = {
        "claims": parsed["claims"],
        "verdict": parsed["verdict"],
        "reason": parsed["reason"],
        "model_called": True,
    }
    print(f"{case_id}: semantic judge glm-4-flash → {parsed['verdict']}，{len(parsed['claims'])} 条 claim")
    for claim in parsed["claims"]:
        print(f"  {claim['relation']}: {claim['claim']} | evidence={claim['evidence_ids']}")

print("语义检查限制：这是独立模型辅助审计，只在固定 4 题和已 hydrate quote 上工作，不能替代人工事实证明或外推到一般任务。")

model_selection_with_intro_scope: semantic judge glm-4-flash → pass，2 条 claim
  supported: 机器学习算法之间没有绝对的优劣之分 | evidence=['evi_065c349a8cbf', 'evi_32c0b230bb7c']
  supported: 机器学习算法的优劣取决于是否适合当前待解决的问题 | evidence=['evi_065c349a8cbf', 'evi_32c0b230bb7c']


cross_validation_reliability: semantic judge glm-4-flash → pass，5 条 claim
  supported: 交叉验证法比单次留出法更可靠。 | evidence=['evi_9a292ab3d933', 'evi_5f5d3dc01c2d']
  supported: 交叉验证法本质上是在进行多次留出法。 | evidence=['evi_9a292ab3d933', 'evi_5f5d3dc01c2d']
  supported: 交叉验证法每次都换不同的子集做测试集。 | evidence=['evi_9a292ab3d933', 'evi_5f5d3dc01c2d']
  supported: 所有样本在交叉验证法中至少做1次测试样本。 | evidence=['evi_9a292ab3d933', 'evi_5f5d3dc01c2d']
  supported: 单次留出法仅依靠1组训练集和测试集对比不同算法效果不够置信，偶然性太强。 | evidence=['evi_833c8fa10a30', 'evi_40e63a429c30']


model_evaluation_and_macro_micro: semantic judge glm-4-flash → pass，5 条 claim
  supported: 模型评估与选择是关于如何评估模型的优劣和选择最适合自己业务场景的模型的过程。 | evidence=['evi_91d0b1e7d0ff']
  supported: 在类别不平衡的情况下，宏平均平等看待每个类别。 | evidence=['evi_ffd64ad64243']
  supported: 宏平均会受到高P和高R类别的影响。 | evidence=['evi_ffd64ad64243']
  supported: 微平均考虑到了每个类别的样本数量。 | evidence=['evi_ffd64ad64243']
  supported: 在样本数量极度不平衡的情况下，数量较多的类别会主导最终结果。 | evidence=['evi_ffd64ad64243']


book_evidence_boundary: semantic judge glm-4-flash → pass，0 条 claim
语义检查限制：这是独立模型辅助审计，只在固定 4 题和已 hydrate quote 上工作，不能替代人工事实证明或外推到一般任务。


## 5. 事后验收：qrels 覆盖、逐字引用与语义拒答

现在才读取 query metadata 和 qrels。对 canonical answerable 题，所有 `essential=true` 的正相关 evidence 必须既被检索到、又被模型引用；对 canonical unanswerable 题，模型必须返回 `insufficient` 且 `citations=[]`，canonical qrels 也必须没有正相关 evidence。

In [6]:
# 生成完成后才读取评测标签；这些字段只用于 post-hoc 验收，不回流模型。
query_meta = {
    row["query_id"]: row
    for row in read_jsonl(DATASET_ROOT / "queries.jsonl")
    if row["query_id"] in CASE_IDS
}
if set(query_meta) != set(CASE_IDS):
    raise AssertionError("canonical query metadata 不完整")
if any("regression" not in row.get("usage", []) for row in query_meta.values()):
    raise AssertionError("capstone 只能使用 canonical regression 问题")

qrels_by_case = {}
for row in read_jsonl(DATASET_ROOT / "qrels.jsonl"):
    if row["query_id"] in CASE_IDS:
        qrels_by_case.setdefault(row["query_id"], []).append(row)

audit_rows = []
for case_id in CASE_IDS:
    metadata = query_meta[case_id]
    qrels = qrels_by_case.get(case_id, [])
    positives = {row["evidence_id"] for row in qrels if int(row["relevance"]) == 1}
    essential = {
        row["evidence_id"]
        for row in qrels
        if int(row["relevance"]) == 1 and bool(row.get("essential"))
    }
    retrieved_set = set(retrieved_ids[case_id])
    context_set = {row["evidence_id"] for row in context_rows[case_id]}
    cited_set = set(responses[case_id]["citation_ids"])
    semantic = semantic_results[case_id]
    if semantic["model_called"] is not True:
        raise AssertionError(f"语义评审没有真实调用模型：{case_id}")
    if metadata["answerability"] == "answerable":
        if responses[case_id]["status"] != "answerable" or not responses[case_id]["model_called"]:
            raise AssertionError(f"answerable 题模型未返回 answerable：{case_id}")
        if not essential:
            raise AssertionError(f"answerable 题没有 essential qrels：{case_id}")
        if not essential <= retrieved_set:
            raise AssertionError(f"essential evidence 未被检索：{case_id} {essential - retrieved_set}")
        if not essential <= context_set:
            raise AssertionError(f"essential evidence 被上下文预算截掉：{case_id} {essential - context_set}")
        if not essential <= cited_set:
            raise AssertionError(f"模型没有引用全部 essential evidence：{case_id} {essential - cited_set}")
        # qrels 是已审定的正相关集合；未被 qrels 标为正的上下文引用不
        # 自动等同于错误，但必须能回到本次上下文，且 essential 证据全覆盖。
        if not cited_set <= context_set:
            raise AssertionError(f"模型引用不在本次上下文中：{case_id}")
        if semantic["verdict"] != "pass" or not semantic["claims"]:
            raise AssertionError(f"回答语义检查未通过：{case_id}")
        if any(claim["relation"] != "supported" for claim in semantic["claims"]):
            raise AssertionError(f"回答包含未被 quote 支持的主要结论：{case_id}")
        if any(not cited_set.issuperset(set(claim["evidence_ids"])) for claim in semantic["claims"]):
            raise AssertionError(f"语义评审使用了回答未引用的 evidence：{case_id}")
        verdict = "pass_answerable"
    else:
        if positives:
            raise AssertionError(f"unanswerable 题存在正相关 qrels：{case_id}")
        if (
            responses[case_id]["status"] != "insufficient"
            or not responses[case_id]["model_called"]
            or responses[case_id]["citation_ids"]
            or responses[case_id]["answer"] != INSUFFICIENT_ANSWER
        ):
            raise AssertionError(f"unanswerable 题没有保持 insufficient/citations=[] 契约：{case_id}")
        if semantic["verdict"] != "pass" or semantic["claims"]:
            raise AssertionError(f"拒答语义检查失败或夹带结论：{case_id}")
        verdict = "pass_insufficient"
    audit_rows.append({
        "query_id": case_id,
        "answerability": metadata["answerability"],
        "retrieved_ids": retrieved_ids[case_id],
        "context_evidence_ids": sorted(context_set),
        "citation_ids": sorted(cited_set),
        "qrels_positive_citation_ids": sorted(cited_set & positives),
        "essential_ids": sorted(essential),
        "response_status": responses[case_id]["status"],
        "answer": responses[case_id]["answer"],
        "model_called": responses[case_id]["model_called"],
        "semantic_verdict": semantic["verdict"],
        "semantic_claims": semantic["claims"],
        "semantic_judge_model_called": semantic["model_called"],
        "verdict": verdict,
    })

for case_id in CASE_IDS:
    response = responses[case_id]
    print(f"\n[{case_id}] {response['status']}")
    print(response["answer"])
    for citation in response.get("hydrated_citations", []):
        print(f"  cite {citation['evidence_id']} | p.{citation['page']} | quote={citation['quote']}")

print("\n验收结果：")
for row in audit_rows:
    print(f"  {row['query_id']}: {row['verdict']}")

emit_tutorial_audit({
    "trace_kind": "c7_endpoint_capstone",
    "model": RAG_LLM_MODEL,
    "retriever": "bm25",
    "evidence_scope": "canonical evidence_type=quote only",
    "top_k": TOP_K,
    "context_budget_chars": CONTEXT_BUDGET,
    "semantic_judge": {
        "model": RAG_LLM_MODEL,
        "calls": len(CASE_IDS),
        "limitation": "独立模型辅助的逐条 quote 蕴含检查，只覆盖固定 4 题，不能替代人工事实证明或外推到一般任务。",
    },
    "cases": audit_rows,
    "all_passed": True,
})


[model_selection_with_intro_scope] answerable
机器学习算法之间没有绝对的优劣之分，只有是否适合当前待解决的问题之分
  cite evi_065c349a8cbf | p.17 | quote=机器学习算法之间没有绝对的优劣之分，只有是否适合当前待解决的问题之分
  cite evi_32c0b230bb7c | p.17 | quote=没有绝对的优劣之分，只有是否适合当前待解决的问题之分

[cross_validation_reliability] answerable
交叉验证法比单次留出法更可靠的原因在于交叉验证法本质上是在进行多次留出法，且每次都换不同的子集做测试集，最终让所有样本均至少做1次测试样本。这样做的理由是因为一般的留出法只会划分出1组训练集和测试集，仅依靠1组训练集和测试集去对比不同算法之间的效果显然不够置信，偶然性太强。通过多次划分，每次采用不同的子集作为测试集，可以减少偶然性，提高评估结果的可靠性。
  cite evi_718a2d52ed5c | p.18 | quote=2.2 评估方法 本节介绍了3 种模型评估方法：留出法、交叉验证法、自助法。留出法由于操作简单，因此最常用； 交叉验证法常用于对比同一算法的不同参数配置之间的效果，以及对比不同算法之间的效果；自助法常用 于集成学习（详见“西瓜书”第8 章的8.2 节和8.3 节）产生基分类器。留出法和自助法简单易懂，在此 不再赘述，下面举例说明交叉验证法的常用方式。 对比同一算法的不同参数配置之间的效果：假设现有数据集D，且有一个被评估认为适合用于数据集 D 的算法L，该算法有可配置的参数，假设备选的参数配置方案有两套：方案a，方案b。下
  cite evi_df934cb5bae0 | p.18 | quote=本节介绍了3 种模型评估方法：留出法、交叉验证法、自助法
  cite evi_9a292ab3d933 | p.19 | quote=从以上的举例可以看出，交叉验证法本质上是在进行多次留出法，且每次都换不同的子集做测试集， 最终让所有样本均至少做1 次测试样本。
  cite evi_5f5d3dc01c2d | p.19 | quote=交叉验证法本质上是在进行多次留出法，且每次都换不同的子集做测试集， 最终让所有样本

## 小结

这个 capstone 只验收一条小而完整的路径：检索结果先经过预算，再由真实模型生成可检查的身份引用，最后用 canonical qrels 做事后审计。它不能代替 C7 的 BM25/CCH 核心 benchmark，也不宣称评估 C2 训练候选或 C6 补充资料。